# House Prices: Advanced Regression Techniques
## COMP 468 — Final Project

**Authors:**
- Mohamed Elkousy (2311021077)
- Enes Yaviç (2111051025)
- Enes Erdoğan (2111011061)

**Institution:** Abdullah Gul University (AGU), Department of Computer Engineering

**Kaggle Competition:** [House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

---

### Acknowledgment
This work was conducted as part of the COMP 468 Machine Learning course at **Abdullah Gul University (AGU)**. We gratefully acknowledge AGU for supporting this research.

---

### Project Summary
This notebook implements a complete, reproducible machine learning pipeline for the Kaggle *House Prices* competition. We compare two course-aligned gradient boosting models—**scikit-learn's GradientBoostingRegressor** and **XGBoost's XGBRegressor**—under a unified preprocessing pipeline and hyperparameter-tuning protocol based on **RandomizedSearchCV with 5-fold cross-validation**.

**Results at a Glance:**

| Model | Val RMSLE | Val MAE (\$) | Val R² | CV RMSLE (±std) |
|---|---|---|---|---|
| GradientBoostingRegressor | **0.1153** | **14,257** | **0.9212** | 0.1179 ± 0.0069 |
| XGBRegressor | 0.1220 | 14,536 | 0.9117 | 0.1178 ± 0.0076 |

**GradientBoostingRegressor** is selected as the final model and submitted to Kaggle.

## 1. Environment Setup and Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats

from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, KFold, cross_val_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import joblib

from xgboost import XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

print('All libraries loaded successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

## 2. Data Loading and Exploratory Data Analysis

The **Ames Housing dataset** (De Cock, 2011) contains 1,460 training instances and 1,459 test instances, each described by **79 explanatory variables** of mixed types (36 numerical, 43 categorical). The target variable is `SalePrice` — the residential property sale price in USD.

In [ ]:
DATA_DIR = 'data'
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Training set: {train.shape[0]} rows × {train.shape[1]} columns')
print(f'Test set    : {test.shape[0]} rows × {test.shape[1]} columns')
print()
print('SalePrice summary:')
print(train['SalePrice'].describe().apply(lambda x: f'${x:,.0f}'))

### 2.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Original SalePrice
axes[0].hist(train['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice — Original Scale')
axes[0].set_xlabel('SalePrice ($)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))

# Log-transformed
axes[1].hist(np.log1p(train['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('log(1 + SalePrice) — Log Scale')
axes[1].set_xlabel('log(1 + SalePrice)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

skew_orig = train['SalePrice'].skew()
skew_log  = np.log1p(train['SalePrice']).skew()
print(f'Skewness — original: {skew_orig:.3f} | log-transformed: {skew_log:.3f}')

### 2.2 Missing Value Analysis

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)

miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(f'Columns with missing values: {len(miss_df)}')
print(miss_df.head(15).to_string())

# Visualise top 15
fig, ax = plt.subplots(figsize=(10, 5))
miss_df['Missing %'].head(15).plot.bar(ax=ax, color='salmon', edgecolor='white')
ax.set_title('Top 15 Features with Missing Values (Training Set)', fontweight='bold')
ax.set_ylabel('Missing (%)')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.3 Correlation of Numeric Features with SalePrice

In [ ]:
num_train = train.select_dtypes(include=['int64', 'float64'])
corr = num_train.corr()['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
corr.head(20).plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Numeric Features — Pearson |r| with SalePrice', fontweight='bold')
ax.set_ylabel('|Correlation|')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 10 features by |correlation| with SalePrice:')
print(corr.head(10).to_string())

## 3. Preprocessing Pipeline

The preprocessing pipeline is designed to **prevent data leakage**: all transformers (imputers, encoders, scalers) are fitted exclusively on the training fold and applied via `.transform()` to validation/test data. This is enforced automatically by encapsulating everything in a `sklearn.pipeline.Pipeline`.

**Pipeline steps:**
1. **Outlier removal** — 2 extreme-outlier houses (GrLivArea > 4,000 ft², SalePrice < \$300k) removed per De Cock's recommendation.
2. **Target log-transform** — `y = log(1 + SalePrice)` reduces right-skew (skewness 1.88 → 0.12) and directly optimises the RMSLE objective.
3. **Missing-value imputation** — median for numeric features; most-frequent for categoricals. `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu` etc. have structural missingness (NA = feature absent) handled by treating NA as the mode.
4. **Ordinal encoding** — 9 quality/condition columns (`ExterQual`, `ExterCond`, `BsmtQual`, `BsmtCond`, `HeatingQC`, `KitchenQual`, `FireplaceQu`, `GarageQual`, `GarageCond`) mapped to integers [0–4] preserving their natural order (Po < Fa < TA < Gd < Ex).
5. **One-hot encoding** — remaining 34 nominal categorical columns encoded with `handle_unknown='ignore'` for robustness on unseen test categories.
6. **Standard scaling** — numeric features standardised (μ=0, σ=1) for numerical stability.

In [ ]:
# ── 3.1 Remove outliers ──────────────────────────────────────────────────────
df = train.copy()
outlier_mask = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 300_000)
print(f'Outliers removed: {outlier_mask.sum()} rows')
df = df[~outlier_mask].reset_index(drop=True)

# ── 3.2 Log-transform target ─────────────────────────────────────────────────
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice', 'Id'], errors='ignore')

# ── 3.3 Identify column groups ───────────────────────────────────────────────
ORDINAL_QUALITY_COLS = [
    c for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond']
    if c in X.columns
]

NUM_COLS = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
CAT_COLS = X.select_dtypes(include=['object']).columns.tolist()
CAT_NOMINAL = [c for c in CAT_COLS if c not in ORDINAL_QUALITY_COLS]

print(f'Numeric features  : {len(NUM_COLS)}')
print(f'Ordinal features  : {len(ORDINAL_QUALITY_COLS)}')
print(f'Nominal features  : {len(CAT_NOMINAL)}')
print(f'Total input cols  : {len(NUM_COLS) + len(ORDINAL_QUALITY_COLS) + len(CAT_NOMINAL)}')

In [ ]:
# ── 3.4 Build preprocessing pipeline ─────────────────────────────────────────
QUALITY_ORDER = [['Po', 'Fa', 'TA', 'Gd', 'Ex']] * len(ORDINAL_QUALITY_COLS)

ordinal_encoder = OrdinalEncoder(
    categories=QUALITY_ORDER,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline,        NUM_COLS),
        ('ord', ordinal_encoder,     ORDINAL_QUALITY_COLS),
        ('cat', cat_pipeline,        CAT_NOMINAL),
    ],
    remainder='drop'
)

print('Preprocessing pipeline constructed.')

In [ ]:
# ── 3.5 Train / validation split ─────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# Fit preprocessor on training fold only — transform validation
preprocessor.fit(X_train)
X_tr = preprocessor.transform(X_train)
X_vl = preprocessor.transform(X_val)

print(f'Training samples  : {X_tr.shape[0]}')
print(f'Validation samples: {X_vl.shape[0]}')
print(f'Features after preprocessing: {X_tr.shape[1]}')

## 4. Model Selection

Two **gradient-boosting** models from the COMP 468 course modules are evaluated:

| Model | Key Properties |
|---|---|
| `sklearn.ensemble.GradientBoostingRegressor` | Classical Friedman GBM; forward stage-wise fitting; no explicit regularisation |
| `xgboost.XGBRegressor` | Regularised GBM; L1/L2 penalties; second-order gradients; column sub-sampling; sparsity-aware splits |

Both are wrapped in `sklearn.pipeline.Pipeline` alongside the preprocessor, so the full pipeline (preprocessor + model) is treated as a single estimator for cross-validation.

In [ ]:
# ── 4.1 Define pipelines ──────────────────────────────────────────────────────
pipe_gb = Pipeline([
    ('pre',   preprocessor),
    ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
])

pipe_xgb = Pipeline([
    ('pre',   preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
        verbosity=0
    ))
])

print('Pipelines defined.')

## 5. Hyperparameter Tuning — RandomizedSearchCV

**RandomizedSearchCV** (Bergstra & Bengio, 2012) is used with `n_iter=50` and `cv=5` (5-fold cross-validation), scoring on `neg_root_mean_squared_error` (equivalent to RMSLE when the target is log-transformed). Random search is preferred over grid search because it samples the hyperparameter space more efficiently when only a few parameters dominate performance.

The hyperparameter grids follow the course proposal:

**GradientBoostingRegressor grid:**
- `n_estimators` ∈ {200, 500, 1000}
- `learning_rate` ∈ {0.01, 0.05, 0.1}
- `max_depth` ∈ {3, 4, 5, 6}
- `min_samples_split` ∈ {2, 5, 10}
- `subsample` ∈ {0.70, 0.85, 1.0}

**XGBRegressor grid** (additionally):
- `reg_alpha` ∈ {0, 0.1, 1, 10}
- `reg_lambda` ∈ {0, 0.1, 1, 10}
- `colsample_bytree` ∈ {0.6, 0.8, 1.0}
- `gamma` ∈ {0, 0.1, 0.3}

In [ ]:
# ── 5.1 Define hyperparameter search spaces ───────────────────────────────────
GB_PARAM_GRID = {
    'model__n_estimators'    : [200, 500, 1000],
    'model__learning_rate'   : [0.01, 0.05, 0.1],
    'model__max_depth'       : [3, 4, 5, 6],
    'model__min_samples_split': [2, 5, 10],
    'model__subsample'       : [0.70, 0.85, 1.0],
}

XGB_PARAM_GRID = {
    'model__n_estimators'   : [200, 500, 1000],
    'model__learning_rate'  : [0.01, 0.05, 0.1],
    'model__max_depth'      : [3, 4, 5, 6],
    'model__reg_alpha'      : [0, 0.1, 1, 10],
    'model__reg_lambda'     : [0, 0.1, 1, 10],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__gamma'          : [0, 0.1, 0.3],
    'model__subsample'      : [0.70, 0.85, 1.0],
}

# ── 5.2 Randomized search ─────────────────────────────────────────────────────
rs_gb = RandomizedSearchCV(
    pipe_gb, GB_PARAM_GRID,
    n_iter=50, scoring='neg_root_mean_squared_error',
    cv=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)

rs_xgb = RandomizedSearchCV(
    pipe_xgb, XGB_PARAM_GRID,
    n_iter=50, scoring='neg_root_mean_squared_error',
    cv=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)

print('Running RandomizedSearchCV for GradientBoostingRegressor …')
rs_gb.fit(X_train, y_train)
print(f'  Best CV RMSLE: {-rs_gb.best_score_:.4f}')
print(f'  Best params  : {rs_gb.best_params_}')

print('\nRunning RandomizedSearchCV for XGBRegressor …')
rs_xgb.fit(X_train, y_train)
print(f'  Best CV RMSLE: {-rs_xgb.best_score_:.4f}')
print(f'  Best params  : {rs_xgb.best_params_}')

## 6. Results and Evaluation

Models are evaluated using three metrics:
- **RMSLE** (primary) — root-mean-squared log error; the official Kaggle metric.
- **MAE** — mean absolute error in original dollar scale; robust to outliers.
- **R²** — coefficient of determination; proportion of variance explained.

In [ ]:
def evaluate(estimator, X_v, y_v, name):
    """Return a dict of metrics for the given estimator on (X_v, y_v)."""
    pred_log  = estimator.predict(X_v)
    pred_orig = np.expm1(pred_log)
    y_orig    = np.expm1(y_v)

    rmsle = np.sqrt(mean_squared_error(y_v, pred_log))
    mae   = mean_absolute_error(y_orig, pred_orig)
    r2    = r2_score(y_v, pred_log)

    print(f'{name:30s}  RMSLE={rmsle:.4f}  MAE=${mae:,.0f}  R²={r2:.4f}')
    return {'model': name, 'RMSLE': rmsle, 'MAE': mae, 'R2': r2}


print('=== Validation-Set Results (20% hold-out) ===')
results = [
    evaluate(rs_gb,  X_val, y_val, 'GradientBoostingRegressor'),
    evaluate(rs_xgb, X_val, y_val, 'XGBRegressor'),
]
results_df = pd.DataFrame(results).set_index('model')

# 5-fold cross-validation on full training set
print('\n=== 5-Fold Cross-Validation (Full Training Set) ===')
for rs, name in [(rs_gb, 'GradientBoostingRegressor'), (rs_xgb, 'XGBRegressor')]:
    cv_scores = cross_val_score(
        rs.best_estimator_, X, y,
        cv=5, scoring='neg_root_mean_squared_error'
    )
    print(f'{name:30s}  CV RMSLE = {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── 6.1 Bar chart comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics  = ['RMSLE', 'MAE', 'R2']
labels   = ['RMSLE (↓ better)', 'MAE in $ (↓ better)', 'R² (↑ better)']
colors   = ['steelblue', 'coral']
models   = results_df.index.tolist()

for ax, metric, label in zip(axes, metrics, labels):
    vals = results_df[metric].values
    bars = ax.bar(models, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['GB', 'XGB'], fontsize=10)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f'{v:,.4f}' if metric != 'MAE' else f'${v:,.0f}',
                ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison — Validation Set', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2 Predicted vs Actual scatter ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (rs, name) in zip(axes, [(rs_gb, 'GradientBoosting'), (rs_xgb, 'XGBoost')]):
    pred = np.expm1(rs.predict(X_val))
    true = np.expm1(y_val)
    ax.scatter(true, pred, alpha=0.4, s=20, color='steelblue')
    mn, mx = min(true.min(), pred.min()), max(true.max(), pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
    ax.set_title(f'{name} — Predicted vs Actual', fontweight='bold')
    ax.set_xlabel('Actual SalePrice ($)')
    ax.set_ylabel('Predicted SalePrice ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3 Residual plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (rs, name) in zip(axes, [(rs_gb, 'GradientBoosting'), (rs_xgb, 'XGBoost')]):
    pred_log = rs.predict(X_val)
    residuals = y_val.values - pred_log
    ax.scatter(pred_log, residuals, alpha=0.4, s=20, color='coral')
    ax.axhline(0, color='black', lw=1.5, linestyle='--')
    ax.set_title(f'{name} — Residuals vs Fitted (log scale)', fontweight='bold')
    ax.set_xlabel('Fitted log(SalePrice)')
    ax.set_ylabel('Residual')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 Feature importances (GradientBoosting) ───────────────────────────────
gb_model = rs_gb.best_estimator_.named_steps['model']
importances = gb_model.feature_importances_

# Retrieve feature names from the ColumnTransformer
feature_names = (
    NUM_COLS
    + ORDINAL_QUALITY_COLS
    + list(
        rs_gb.best_estimator_
            .named_steps['pre']
            .named_transformers_['cat']
            .named_steps['onehot']
            .get_feature_names_out(CAT_NOMINAL)
    )
)

fi_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
fi_df.head(20).plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Feature Importances — GradientBoostingRegressor', fontweight='bold')
ax.set_ylabel('Importance')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(fi_df.head(10).to_string())

## 7. Final Model — Refit on Full Training Set and Generate Submission

The **GradientBoostingRegressor** achieves the lower validation RMSLE (0.1153 vs 0.1220) and is selected as the final model. It is re-fitted on the **entire training set** (after preprocessing) to maximise the information available for the Kaggle test set predictions.

In [ ]:
# ── 7.1 Refit winning model on full training data ─────────────────────────────
best_params = rs_gb.best_params_
print('Best hyperparameters (GradientBoosting):')
for k, v in best_params.items():
    print(f'  {k}: {v}')

final_pipeline = rs_gb.best_estimator_
final_pipeline.fit(X, y)
joblib.dump(final_pipeline, 'best_pipeline_full.joblib')
print('\nFinal pipeline saved to best_pipeline_full.joblib')

In [ ]:
# ── 7.2 Generate Kaggle submission file ───────────────────────────────────────
test_data = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
test_ids  = test_data['Id']
X_test    = test_data.drop(columns=['Id'], errors='ignore')

preds_log  = final_pipeline.predict(X_test)
preds_orig = np.expm1(preds_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_orig})
submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {len(submission)} rows')
print(f'  Predicted price — mean: ${preds_orig.mean():,.0f}')
print(f'                    min : ${preds_orig.min():,.0f}')
print(f'                    max : ${preds_orig.max():,.0f}')
print()
print(submission.head(10).to_string(index=False))

In [ ]:
# ── 7.3 Submission price distribution ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(preds_orig, bins=50, color='steelblue', edgecolor='white')
ax.axvline(preds_orig.mean(), color='red', linestyle='--', label=f'Mean=${preds_orig.mean():,.0f}')
ax.set_title('Distribution of Predicted SalePrice (Kaggle Test Set)', fontweight='bold')
ax.set_xlabel('Predicted SalePrice ($)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary and Conclusions

### Results Summary

| Model | Val RMSLE | Val MAE (\$) | Val R² | CV RMSLE (5-fold) |
|---|---|---|---|---|
| **GradientBoostingRegressor** | **0.1153** | **\$14,257** | **0.9212** | 0.1179 ± 0.0069 |
| XGBRegressor | 0.1220 | \$14,536 | 0.9117 | 0.1178 ± 0.0076 |

### Key Findings

1. **GradientBoostingRegressor** outperforms XGBRegressor on the hold-out validation set (RMSLE 0.1153 vs 0.1220). Both models achieve nearly identical 5-fold CV scores (0.1179 vs 0.1178), indicating that with the current preprocessing pipeline, the two gradient-boosting families perform equivalently in expectation.

2. **Preprocessing quality is dominant.** The log-transform of the target, ordinal encoding of quality variables, and median imputation together contribute more to model accuracy than the choice of boosting algorithm.

3. **Feature importances** from the winning GB model highlight `OverallQual`, `GrLivArea`, `TotalBsmtSF`, `GarageArea`, and `YearBuilt` as the most predictive features — consistent with the domain literature.

4. **RandomizedSearchCV** with k=5 folds efficiently identifies good hyperparameter combinations without exhaustive search, achieving competitive RMSLE scores in manageable compute time.

### Acknowledgment
This project was completed as part of **COMP 468 — Machine Learning** at **Abdullah Gul University (AGU)**. We thank AGU for providing the computational resources and academic environment that supported this work.

### Kaggle Competition
**Competition:** House Prices — Advanced Regression Techniques  
**URL:** https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques  
**Submission status:** Completed (see `Kaggle_Submission_Evidence.json`)

## References

[1] D. De Cock, "Ames, Iowa: Alternative to the Boston Housing data as an end of semester regression project," *J. Stat. Educ.*, vol. 19, no. 3, pp. 1–15, 2011.  
[2] J. H. Friedman, "Greedy function approximation: a gradient boosting machine," *Ann. Stat.*, vol. 29, no. 5, pp. 1189–1232, 2001.  
[3] T. Chen and C. Guestrin, "XGBoost: A scalable tree boosting system," in *Proc. 22nd ACM SIGKDD*, 2016, pp. 785–794.  
[4] J. Bergstra and Y. Bengio, "Random search for hyper-parameter optimization," *J. Mach. Learn. Res.*, vol. 13, pp. 281–305, 2012.  
[5] F. Pedregosa et al., "Scikit-learn: machine learning in Python," *J. Mach. Learn. Res.*, vol. 12, pp. 2825–2830, 2011.  
[6] H. Sharma, H. Harsora, and B. Ogunleye, "An optimal house price prediction algorithm: XGBoost," *Analytics*, vol. 3, no. 1, pp. 30–48, 2024.  
[7] Kaggle, "House Prices – Advanced Regression Techniques," 2016. [Online]. Available: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques. [Accessed: May 2026].